# Cell frames

The block is stored once, in its own frame and the table moves that frame.
Here we check that goin from the block frame to the world and back gives the same points and we look at the cell in Swift with a pawn made from a radius profile (`proxy_pawn`) in place of the statue. 

## Setup

In [1]:
import logging
import warnings

# A few docstrings in the toolbox trigger SyntaxWarnings when Pytohn compiles them... not related to our code.
warnings.filterwarnings("ignore", category=SyntaxWarning)

import numpy as np
import pandas as pd
import roboticstoolbox as rtb
import spatialgeometry as sg
import swift
from sculptor.cell import Cell
from sculptor.model import proxy_pawn
from sculptor.scene import add_block, add_chisel, add_table, fix_mesh_paths, place_block

# Closing a Swift window leaves a few websocket task pending but they are harmless
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

# Without this, Swift on Windows cannot load any mesh (see sculptor/scene.py)
fix_mesh_paths()

## 1. Block frame and world

A point of the block is in the world at

$$\mathbf{p}^W = \mathbf{T}^W_T\,\mathbf{R}_z(\varphi)\,\mathbf{p}^B$$

so going to the world and back has to give the same point at any angle of the table.

In [2]:
# Height of the pawn for now; the real one is chosen in the next notebooks, from what the arm can reach
PAWN_HEIGHT = 0.42
# The round trip only has floating point error, many orders of magnitude below a cube
TOLERANCE = 1e-12
N_ANGLES = 8
QUARTER_TURNS = np.arange(0, 360, 90)

In [3]:
cell = Cell()
voxels = proxy_pawn(height=PAWN_HEIGHT)

angles = np.linspace(0, 2 * np.pi, N_ANGLES, endpoint=False)
errors = [np.abs(cell.world_to_block(cell.block_to_world(voxels.centers, phi), phi) - voxels.centers).max()
          for phi in angles]
assert max(errors) < TOLERANCE
print(f"Largest round-trip error: {max(errors):.1e} m")

Largest round-trip error: 1.9e-16 m


In [4]:
# A corner of the block during a turn of the table
corner = voxels.centers[np.argmax(voxels.centers[:, 0] + voxels.centers[:, 1])]
print(f"Corner in the block frame: {np.round(corner, 3)}")

rows = []
for deg in QUARTER_TURNS:
    p = cell.block_to_world(corner, np.deg2rad(deg))[0]
    rows.append({"phi [deg]": deg, "x [m]": p[0], "y [m]": p[1], "z [m]": p[2],
                 "radius [m]": np.hypot(*(p[:2] - cell.table_top.t[:2]))})
pd.DataFrame(rows).set_index("phi [deg]").round(3)

Corner in the block frame: [0.15  0.15  0.015]


,x [m],y [m],z [m],radius [m]
phi [deg],,,,
0,0.9,0.15,0.215,0.212
90,0.6,0.15,0.215,0.212
180,0.6,-0.15,0.215,0.212
270,0.9,-0.15,0.215,0.212


The round trip is exact up to floating point.   
The corner stays at $z = 0.215m$ and at $0.212m$ from the axis of the table, and each quarter turn moves it to the next corner around that axis, so the table only turns the block about tis own center.

## 2. Scene in Swift

The robot at its ready pose, the table and the block, with the statue in a darker color. The axes show the world frame at the base of the robot, the block frame and the chisel tip.

The table makes one turn. Swift draws in the browser, so this part leaves no output in the notebook.

In [5]:
# One turn of the table in Swift, in steps of DT seconds
DT = 0.05
N_STEPS = 80

env = swift.Swift()
env.launch(realtime=True, browser=None)

panda = rtb.models.Panda()
panda.q = panda.qr
env.add(panda)
add_table(env, cell)
cubes, ids = add_block(env, voxels, cell)

T_tip = panda.fkine(panda.q, tool=cell.tip)
chisel = add_chisel(env, T_tip)
env.add(sg.Axes(length=0.15))
block_axes = sg.Axes(length=0.15, pose=cell.block_pose(0.0))
env.add(block_axes)
env.add(sg.Axes(length=0.08, pose=T_tip))

for k in range(N_STEPS + 1):
    phi = 2 * np.pi * k / N_STEPS
    place_block(cubes, voxels, cell, phi)
    block_axes.T = cell.block_pose(phi)
    env.step(DT)

## Summary

The next notebooks reuse `Cell` from `src/sculptor/cell.py`, with the table top (`TABLE_TOP`) and the chisel tip (`TIP`) defined there, and the drawing helpers in `src/sculptor/scene.py`, which place the cubes with the same `block_pose`.

The round trip between the block frame and the world is exact up to floating point (section 1), and turning the table moves the block only around the axis of the table (the corner table in section 1).